# Notebook 035: Delta Upsert and MERGE INTO Mastery with Multiple Conditions

**Module:** Module 04: Delta Lake Architecture, ACID Operations & Time Travel  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 17: Lakeflow Declarative Pipelines & Video 26: Retail Data Engineering Project  

---

## 1. Business Scenario & Real-World Context
A retail inventory management system synchronizes daily inventory updates from 500 physical warehouses. If an item exists in the central inventory table, update stock quantity and last_restocked date; if missing, insert the new SKU; if stock drops to -999 (discontinued), delete the item.

## 2. Technical Requirement & Problem Statement
Master the `MERGE INTO` API in Delta Lake: implement complex upserts with `whenMatchedUpdate`, `whenMatchedDelete`, `whenNotMatchedInsert`, and conditional update clauses to handle real-world CDC synchronization in a single atomic transaction.

## 3. Spark & Databricks Architecture Notes
Delta MERGE is an atomic operation that performs a full outer join between source updates and target Delta files to identify matching keys. It rewrites only the specific files containing matching rows, avoiding full table rewrites.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_035_Delta_Upsert_and_MERGE_INTO_Mastery_with_Multiple_Conditions")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Initializing Current Warehouse Inventory (Target Delta Table)
Create target inventory table.

In [ ]:
from delta.tables import DeltaTable

delta_inv_path = "/tmp/delta_inventory_035"
if Path(delta_inv_path).exists():
    shutil.rmtree(delta_inv_path)

current_inventory = [
    ("SKU-101", "Wireless Mouse", 50, 24.99, "2023-11-01"),
    ("SKU-102", "Mechanical Keyboard", 20, 89.99, "2023-11-01"),
    ("SKU-103", "USB-C Hub", 15, 35.00, "2023-11-01"),
    ("SKU-104", "Obsolete Cable", 5, 4.99, "2023-10-01")
]

schema = "sku STRING, product_name STRING, quantity INT, unit_price DOUBLE, last_updated STRING"
spark.createDataFrame(current_inventory, schema).write.format("delta").mode("overwrite").save(delta_inv_path)

target_table = DeltaTable.forPath(spark, delta_inv_path)
print("Initial Target Inventory:")
target_table.toDF().show()

## 6. Preparing Inbound Daily Warehouse Updates (Source Batch)
Create batch with updates, new SKUs, and a discontinued item (quantity = -999).

In [ ]:
incoming_updates = [
    ("SKU-101", "Wireless Mouse", 80, 24.99, "2023-11-20"),      # Update quantity to 80
    ("SKU-104", "Obsolete Cable", -999, 0.00, "2023-11-20"),      # Discontinued -> Delete
    ("SKU-105", "Noise Cancelling Headset", 40, 199.99, "2023-11-20") # New SKU -> Insert
]

df_updates = spark.createDataFrame(incoming_updates, schema)
print("Incoming Daily Updates Batch:")
df_updates.show()

## 7. Executing Multi-Condition MERGE INTO
Execute conditional update, conditional delete, and insert in a single atomic pass.

In [ ]:
(
    target_table.alias("target")
    .merge(
        source=df_updates.alias("source"),
        condition="target.sku = source.sku"
    )
    .whenMatchedDelete(
        condition="source.quantity = -999"
    )
    .whenMatchedUpdate(
        set={
            "quantity": F.col("source.quantity"),
            "unit_price": F.col("source.unit_price"),
            "last_updated": F.col("source.last_updated")
        }
    )
    .whenNotMatchedInsert(
        values={
            "sku": F.col("source.sku"),
            "product_name": F.col("source.product_name"),
            "quantity": F.col("source.quantity"),
            "unit_price": F.col("source.unit_price"),
            "last_updated": F.col("source.last_updated")
        }
    )
    .execute()
)

print("Target Table after MERGE INTO:")
target_table.toDF().orderBy("sku").show()

## 9. Key Architectural Takeaways
- **`MERGE INTO` executes inserts, updates, and deletes in a single unified atomic transaction.**
- **Always ensure the source dataset is deduplicated on the merge condition key; duplicate source keys will throw a `DeltaAnalysisException`.**
- **Specify partition pruning filters in the merge condition whenever merging into partitioned Delta tables to skip untouched partitions.**

## 10. Senior Data Engineering Interview Questions
1. What happens when the source table in a Delta `MERGE` statement contains duplicate keys?
2. How does Delta Lake's merge operation identify which Parquet files need to be rewritten?
3. How can you optimize a MERGE operation on a multi-terabyte partitioned Delta table?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
